<div style="background:linear-gradient(135deg,#1b1448 0%,#3b1f5e 55%,#7c2d12 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#fed7aa;font-weight:700;text-transform:uppercase">Chapter 4 · Solutions</div>
  <div style="font-size:34px;font-weight:900;line-height:1.1;margin:10px 0 6px">Your first model call · Solutions</div>
  <div style="font-size:15px;color:#ffe4d6;max-width:740px;line-height:1.6">Worked answers to the chapter's three practice challenges.</div>
  <div style="font-size:13px;color:#fed7aa;margin-top:14px">Building AI with AI: A Visual Handbook · John Fisher</div>
</div>

Worked answers to Chapter 4's practice challenges. The first two call the model through the chapter's `ask.py`, and the third tests `ask.py` with a stand-in client.

## Setup

The same Setup as the chapter's notebook, with this notebook's own recording, and the client from the chapter's Claude Code build.

In [1]:
# Set LIVE = True to send these requests to the Anthropic API with your own key, saved in
# Colab's secrets panel as ANTHROPIC_API_KEY. Left False, the notebook replays the answers
# recorded for the book, with no key and no cost.
LIVE = False
%pip install -q --disable-pip-version-check anthropic==2.7.0
import importlib, pathlib, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/chapters/notebooks/"
    pathlib.Path("recordings").mkdir(exist_ok=True)
    for name in ["book_client.py", "recordings/your-first-model-call--solutions.json"]:
        urllib.request.urlretrieve(BASE + name, name)
import book_client
importlib.reload(book_client)
client = book_client.make_client("your-first-model-call--solutions", mode="live" if LIVE else None)

Note: you may need to restart the kernel to use updated packages.


Replaying 4 recorded responses from claude-sonnet-5-5, recorded 2026-11-12. Set LIVE = True in Setup to call the API yourself.


In [2]:
# ask.py is the command-line client Claude Code generated in this chapter's build, from the spec and tests
# in builds/your-first-model-call/. Colab fetches it from the book's repository.
BUILD = "your-first-model-call"
if "google.colab" in sys.modules:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/builds/"
                               f"{BUILD}/generated/ask.py", "ask.py")
else:
    sys.path.insert(0, str(pathlib.Path("../../builds", BUILD, "generated").resolve()))
import ask
ask = importlib.reload(ask)
from ask import answer_text, build_request, chat, format_usage, main

# The whole module, so you can read every function before the notebook uses it
import inspect
print(inspect.getsource(ask))
import io

import argparse
import sys

DEFAULT_MODEL = "claude-sonnet-5-5"
DEFAULT_MAX_TOKENS = 1024


def build_request(prompt, history=None, system=None, model=DEFAULT_MODEL,
                  max_tokens=DEFAULT_MAX_TOKENS):
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt is empty")
    if max_tokens < 1:
        raise ValueError("max_tokens must be at least 1")
    history = list(history) if history else []
    for i, m in enumerate(history):
        expected = "user" if i % 2 == 0 else "assistant"
        role = m.get("role") if isinstance(m, dict) else None
        if role != expected:
            raise ValueError("invalid history")
    if history and history[-1]["role"] != "assistant":
        raise ValueError("history must end with an assistant message")
    request = {"model": model, "max_tokens": max_tokens,
               "messages": history + [{"role": "user", "content": prompt}]}
    if system:
        request["system"] = system
    return re

## Challenge 1. A system prompt that holds across turns

Write a system prompt that makes the support assistant answer in Spanish, then hold a two-turn conversation in English with `chat`.

In [3]:
# Challenge 1. A system prompt that holds across turns.
SYSTEM = "You are a support assistant. Always reply in Spanish, in one or two sentences."
history = chat(client, ["My order arrived with a cracked screen.", "How long does a replacement take?"],
               sys.stdout, system=SYSTEM, show_usage=True)
print(len(history), "messages in the conversation")

Lamento mucho que tu pedido haya llegado con la pantalla rota. Por favor, envíame tu número de pedido y una foto del daño para gestionar de inmediato un reemplazo o reembolso.
[48 tokens in, 73 out]

El tiempo de un reemplazo depende de la disponibilidad del producto y de tu ubicación, por lo que no puedo darte un plazo exacto todavía. Si me compartes tu número de pedido y tu código postal, podré confirmarte una fecha estimada de entrega una vez aprobado el reemplazo.
[132 tokens in, 201 out, 92 of them thinking]

4 messages in the conversation


Both replies came back in Spanish, the first asking for the order number and a photo of the damage, the second explaining that the replacement time depends on stock and location. The system prompt travels with every request, so the second turn kept the rule without the user repeating it. The conversation ended with four messages, two questions and two replies.

## Challenge 2. Examples that pick a category

Write two worked examples that make the model answer with one of Chapter 1's ticket categories, and run them on two new tickets.

In [4]:
# Challenge 2. Worked examples that steer a ticket to one of Chapter 1's categories.
INSTRUCTION = ("Sort this support ticket into one category from billing, login, shipping, refund or other.\n\n"
               "Ticket: ")
EXAMPLES = [
    {"role": "user", "content": INSTRUCTION + "I was charged twice for the same order."},
    {"role": "assistant", "content": "billing"},
    {"role": "user", "content": INSTRUCTION + "The password reset email never arrives."},
    {"role": "assistant", "content": "login"},
]
for ticket in ["My parcel has been stuck in transit for nine days.", "Do you sell gift wrapping?"]:
    message = client.messages.create(**build_request(INSTRUCTION + ticket, history=EXAMPLES))
    print(f"{answer_text(message)!r:12} {ticket}")

'shipping'   My parcel has been stuck in transit for nine days.
'other'      Do you sell gift wrapping?


The parcel stuck in transit came back as `shipping` and the question about gift wrapping as `other`, each a single word from the list. The examples showed `billing` and `login`, and the model still chose labels the examples never used, because the instruction names all five.

## Challenge 3. Find the gap in `--chat`

With a prompt on the command line, `--chat` should answer the prompt before it waits for the next line. Log when standard input is read and when each question is sent, then write the test that requires the prompt to go first.

In [5]:
# Challenge 3. Does --chat send the first question before it reads the next line?
from types import SimpleNamespace

events = []

class Lines:
    """Standard input that logs each line as it is read."""
    def __init__(self, lines):
        self.lines = iter(lines)
    def __iter__(self):
        return self
    def __next__(self):
        line = next(self.lines)
        events.append(f"read {line.strip()!r}")
        return line

class LoggingClient:
    """A stand-in client that logs each question and answers from a fixed reply."""
    def __init__(self):
        self.messages = self
    def create(self, **request):
        events.append(f"sent {request['messages'][-1]['content']!r}")
        return SimpleNamespace(content=[SimpleNamespace(type="text", text="ok")],
                               usage=SimpleNamespace(input_tokens=1, output_tokens=1))

def test_chat_sends_the_prompt_before_reading_stdin():
    events.clear()
    main(["--chat", "Hi"], client=LoggingClient(), stdin=Lines(["Second\n", "Third\n"]), out=io.StringIO())
    assert events[0] == "sent 'Hi'", events

main(["--chat", "Hi"], client=LoggingClient(), stdin=Lines(["Second\n", "Third\n"]), out=io.StringIO())
print("order of events:", events)
try:
    test_chat_sends_the_prompt_before_reading_stdin()
    print("pass")
except AssertionError:
    print("FAIL: main reads all of standard input before it sends the prompt")

order of events: ["read 'Second'", "read 'Third'", "sent 'Hi'", "sent 'Second'", "sent 'Third'"]
FAIL: main reads all of standard input before it sends the prompt


The log shows both lines of standard input read before the prompt was sent. `main` builds its list of lines with `[args.prompt] + list(stdin)`, and `list` reads to the end of the input first. At a terminal that means the first answer waits until the user ends the input with Ctrl-D. The test fails under the generated code. Adding it to the build's `given/tests/`, with a spec sentence saying the prompt is sent first, makes the next build fix it, with `itertools.chain` in place of the list.

---
<div style="text-align:center;color:#666f8e;font-size:12px;margin-top:10px">Building AI with AI: A Visual Handbook · © 2026 John Fisher · text CC BY 4.0, code MIT</div>